# Tutorial: Knowledge Graph Federado

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao Python path para permitir imports do knowledge_graph
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"✓ Diretório {project_root} adicionado ao Python path")
print(f"✓ Agora é possível importar módulos do knowledge_graph")

In [1]:
from knowledge_graph.federator  import FederadorOntologias
from knowledge_graph.ingester   import IngesterMarkdown
from knowledge_graph.detector   import DetectorEntidadesNovas
from knowledge_graph.validator  import ValidadorOntologia
from knowledge_graph.visualizer import VisualizadorKG

ModuleNotFoundError: No module named 'knowledge_graph'

## 0. Fedoração e Inicialização
Nesse exemplo inicial, utilizaremos a base local do rdflib para não depender do endpoint QLever, o que é ótimo para protótipos e validação de dados curtos.

In [ ]:
federador = FederadorOntologias(use_local_rdflib=True, base_dir='..')
federador.carregar_todos_os_grafos()

## 1. Ingestão (Extração de Triplas com LLM)

In [ ]:
import os
# IMPORTANTE: Em um ambiente real é necessário OPENAI_API_KEY no .env.
# Se não possuir a chave na execução, a extração retornará erro ou mock vazio.
# os.environ["OPENAI_API_KEY"] = "sk-..."

ingester = IngesterMarkdown("dados_exemplo/contexto_cadeia_valor.md", base_dir='..')
chunks   = ingester.carregar_chunks()
triplas  = []
for chunk in chunks:
    try: 
        triplas.extend(ingester.extrair_triplas(chunk))
    except Exception as e:
        print(f"Não foi possível processar (API Key não configurada provavelmente): {e}")

triplas_mapeadas = [ingester.mapear_para_ontologia(t) for t in triplas]
federador.inserir_triplas(triplas_mapeadas)

## 2. Detecção de Candidatos


In [ ]:
detector   = DetectorEntidadesNovas(federador)
candidatos = detector.identificar_e_marcar(triplas_mapeadas)
relatorio  = detector.gerar_relatorio_candidatos()
print("---- Relatório de Candidaturas ----")
print(relatorio)

## 2.4 Interface de Validação

In [ ]:
validador = ValidadorOntologia(federador)
# Simulação de aprovação manual de um subconjunto de itens:
if not relatorio.empty:
    for idx, row in relatorio.iterrows():
        # Exemplo automático (Na prática, 'input' humano)
        print(f"Aprovando automaticamente {row['nome_entidade']} para simplificar...")
        validador.aprovar_entidade(row['uri'], aprovador="TutorialBot", tipo_ontologico="http://phd-cesar-rag/kg#Biotecnologia")
        
validador.exportar_nova_versao_ontologia(versao="1.1.0", caminho_saida="../knowledge_graph/ontologias/kg_custom_v1.1.0.ttl")

## 3. Visualização Interativa


In [ ]:
viz = VisualizadorKG(federador)
viz.gerar_html_interativo("../kg_viz.html")
from IPython.display import IFrame
IFrame(src='../kg_viz.html', width='100%', height='600px')